# Poisson Pressure Solver with Variable Density

Let $(x,y,z)\in]x_{\text{min}}, x_{\text{max}}[\times]y_{\text{min}}, y_{\text{max}}[\times]z_{\text{min}}, z_{\text{max}}]=]0, 2\pi[^3=\Omega$, 
then the Poisson's problem is defined as
\begin{equation}
    \begin{split}
        \nabla^2p(x,y,z) -\dfrac{1}{\rho(x,y,z)}\nabla\rho(x,y,z)\cdot\nabla p(x,y,z)&= 
            f(x,y,z) \quad \text{in} ~ \Omega,\\
        \frac{\partial p}{\partial z} &=0, \quad \text{on} ~ z=z_{\text{min}}, \\
        p &= 0, \quad \text{on} ~ z=z_{\text{max}},
    \end{split}
\end{equation}

with periodic boundary conditions in $x$ and $y$ directions.
The solution of this problem is
\begin{equation}
    p(x,y,z) = g(z)G(x,y,z),
\end{equation}

with

\begin{equation*}
    G(x,y,z)=\exp\left(-\left(\frac{(x - x_0)^2}{s_x} + \frac{(y - y_0)^2}{s_y}+ \frac{(z - z_0)^2}{s_z}\right)\right), \quad
    g(z) = \frac{1}{\pi^3}(2\pi z^2-z^3).
\end{equation*}

and

<!--
\begin{equation*}
    g(z) = \frac{1}{\pi^3}(2\pi z^2-z^3).
\end{equation*} 

\begin{equation}
    f(x,y,z) = \frac{1}{\pi^3}
        \left[
            \left(z^3-2\pi z^2\right)
            \left(
            \dfrac{2}{s_x}\left(1-\dfrac{2}{s_x}(x-x_0)^2\right) + 
            \dfrac{2}{s_y}\left(1-\dfrac{2}{s_y}(y-y_0)^2\right)
            \right) 
            + g(z)
        \right]
        G(x,y,z),
\end{equation}
We use $x_0=y_0=z_0=\pi$, and $s_x=s_y=s_z=.5$.
-->

In [1]:
import time as time
from fftfd import *

In [2]:
def fftfd_density(f: np.ndarray, rho:np.ndarray, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> np.ndarray:
    Nx, Ny, Nz = x.shape[0], y.shape[0], z.shape[0]
    dx, dy, dz = x[1] - x[0], y[1] - y[0], z[1] - z[0]
    x_max = x[-1] # Max x value
    y_max = y[-1] # Max y value
    p_bot = np.zeros((Nx, Ny)) # Bottom boundary condition
    p_top = np.zeros((Nx, Ny)) # Top boundary condition
    F = f[:-1, :-1, :-1] # Removing boundaries and top boundary condition
    rr = np.fft.fftfreq(Nx - 1) * (Nx - 1) 
    ss = np.fft.fftfreq(Ny - 1) * (Ny - 1) 
    # Scale frequencies for any domain
    kx = 2 * np.pi * rr * 1j / x_max
    ky = 2 * np.pi * ss * 1j / y_max
    # kx = 2 * np.pi * rr / x_max
    # ky = 2 * np.pi * ss / y_max
    # Compute FFT in x direction (column-wise)
    F_k = np.fft.fft2(F, axes=(0, 1))
    P_k = np.zeros_like(F_k)
    # Compute gradient of density
    rho_x = np.gradient(rho, dx, axis=0, edge_order=2)
    rho_y = np.gradient(rho, dy, axis=1, edge_order=2)
    rho_z = np.gradient(rho, dz, axis=2, edge_order=2)
    # Remove boundaries and top boundary
    # rho_bk = rho
    rho = rho[:,:, 1:-1]
    rho_x = rho_x[:,:,1:-1]
    rho_y = rho_y[:,:,1:-1]
    rho_z = rho_z[:,:,1:-1]
    # rho = rho.astype(np.complex128)
    # rho_x = rho_x.astype(np.complex128)
    # rho_y = rho_y.astype(np.complex128)
    # rho_z = rho_z.astype(np.complex128)
    # Compute FFT in the last row (top boundary condition)
    P_k0 = np.fft.fft2(p_bot, axes=(0, 1))
    P_kNz = np.fft.fft2(p_top, axes=(0, 1))
    b = np.zeros(Nz - 1, dtype=np.complex128)
    c = np.zeros(Nz - 2, dtype=np.complex128)
    # Solve the system for each gamma
    for r in range(Nx - 1):
        for s in range(Ny - 1):
            # Compute gammas
            gamma_right = 1 / dz ** 2 - rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            gamma_left  = 1 / dz ** 2 + rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            gamma_rs = kx[r] ** 2 + ky[s] ** 2 - 2 / dz ** 2 - (kx[r] * rho_x[r, s] + ky[s] * rho_y[r, s]) / rho[r, s]
            # gamma_rs =  -(kx[r] ** 2 + ky[s] ** 2 + 2 / dz ** 2 + (kx[r] * rho_x[r, s] * 1j + ky[s] * rho_y[r, s] * 1j) / rho[r, s])
            # Create RHS of system
            F_k[r, s, 0] = dz * P_k0[r, s] + 0.5 * F_k[r, s, 1] / gamma_right[0] # dp/dz = 0
            # Substract coefficient of top boundary condition
            F_k[r, s, -1] -=  P_kNz[r, s] * gamma_right[-1]
            # Create A in the system. Only keep diagonals of matrix
            a = gamma_left
            b[1:] = gamma_rs * 0 + 1
            c[1:] = gamma_right[:-1]
            # Fix first coefficients
            b[0] = 0.5 * gamma_left[0] / gamma_right[0] - 1.5
            c[0] = 2 + 0.5 * gamma_rs[0] / gamma_right[0]
            # up = 4 * rho_bk[r, s, 1] + rho_bk[r, s, 2] - rho_bk[r, s, 0]
            # do = 4 * rho_bk[r, s, 1] + rho_bk[r, s, 0] - rho_bk[r, s, 2]
            # b[0] = 0.5 * up / do - 1.5
            # tmp1 = gamma_left[0] / gamma_right[0]
            # tmp2 = up / do
            # err_tmp = np.abs(tmp1 - tmp2)
            # if err_tmp > 1e-10:
            #     print(tmp1, tmp2, err_tmp)
            # print(a.shape, b.shape, c.shape, gamma_left.shape, gamma_right.shape, gamma_rs.shape)
            # Solve system A P_k = F_k
            P_k[r, s, :] = thomas_algorithm(a, b, c, F_k[r, s, :])
    # Compute IFFT in x-y slices to restore the 'physical' pressure
    p = np.real(np.fft.ifft2(P_k, axes=(0, 1)))
    # Add x boundary condition
    p = np.concatenate([p, np.expand_dims(p[0], axis=0)], axis=0)
    # Add y boundary condition
    p = np.concatenate([p, np.expand_dims(p[:, 0], axis=1)], axis=1)
    # Add top boundary condition
    p = np.concatenate([p, np.expand_dims(p_top, axis=2)], axis=2)
    return p

In [ ]:
def fftfd_density(f: np.ndarray, rho:np.ndarray, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> np.ndarray:
    Nx, Ny, Nz = x.shape[0], y.shape[0], z.shape[0]
    dx, dy, dz = x[1] - x[0], y[1] - y[0], z[1] - z[0]
    x_max = x[-1] # Max x value
    y_max = y[-1] # Max y value
    p_bot = np.zeros((Nx, Ny)) # Bottom boundary condition
    p_top = np.zeros((Nx, Ny)) # Top boundary condition
    F = f[:-1, :-1, :-1] # Removing boundaries and top boundary condition
    rr = np.fft.fftfreq(Nx - 1) * (Nx - 1) 
    ss = np.fft.fftfreq(Ny - 1) * (Ny - 1) 
    # Scale frequencies for any domain
    kx = 2 * np.pi * rr * 1j / x_max
    ky = 2 * np.pi * ss * 1j / y_max
    kx = 2 * np.pi * rr / x_max
    ky = 2 * np.pi * ss / y_max
    # Compute FFT in x direction (column-wise)
    F_k = np.fft.fft2(F, axes=(0, 1))
    P_k = np.zeros_like(F_k)
    # Compute gradient of density
    rho_p = rho
    # rho_p = rho_p[:-1, :-1, :-1]
    rho_x = np.gradient(rho, dx, axis=0, edge_order=2)
    rho_y = np.gradient(rho, dy, axis=1, edge_order=2)
    rho_z = np.gradient(rho, dz, axis=2, edge_order=2)
    # rho_x = (np.roll(rho_p, -1, axis=0) - np.roll(rho_p, 1, axis=0)) / (2 * dx)
    # rho_y = (np.roll(rho_p, -1, axis=1) - np.roll(rho_p, 1, axis=1)) / (2 * dy)
    # rho_z = (np.roll(rho_p, -1, axis=2) - np.roll(rho_p, 1, axis=2)) / (2 * dz)
    # rho_x = -(np.roll(rho_p, -1, axis=0) - rho_p) / dx
    # rho_y = -(np.roll(rho_p, -1, axis=1) - rho_p) / dy
    # rho_z = -(np.roll(rho_p, -1, axis=2) - rho_p) / dz
    # Repeat periodic boundary conditions
    # rho_x = np.concatenate([rho_x, np.expand_dims(rho_x[0], axis=0)], axis=0)
    # rho_y = np.concatenate([rho_y, np.expand_dims(rho_y[:, 0], axis=1)], axis=1)
    # rho_z = np.concatenate([rho_z, np.expand_dims(rho_z[:, :, 0], axis=2)], axis=2)
    # rho_x = np.ones_like(rho) * 2
    # rho_y = np.ones_like(rho) * 2
    # rho_z = np.ones_like(rho) * 2
    # Remove boundaries and top boundary
    # rho_bk = rho
    # rho = rho[:,:, 1:-1]
    # rho_x = rho_x[:,:,1:-1]
    # rho_y = rho_y[:,:,1:-1]
    # rho_z = rho_z[:,:,1:-1]
    rho = rho[:-1,:-1,1:-1]
    rho_x = rho_x[:-1,:-1,1:-1]
    rho_y = rho_y[:-1,:-1,1:-1]
    rho_z = rho_z[:-1,:-1,1:-1]
    # rho =   rho[1:,1:,:-2]
    # rho_x = rho#_x[1:,1:,:-2]
    # rho_y = rho#_y[1:,1:,:-2]
    # rho_z = rho#_z[1:,1:,:-2]
    # rho = rho.astype(np.complex128)
    # rho_x = rho_x.astype(np.complex128)
    # rho_y = rho_y.astype(np.complex128)
    # rho_z = rho_z.astype(np.complex128)
    # Compute FFT in the last row (top boundary condition)
    P_k0 = np.fft.fft2(p_bot, axes=(0, 1))
    P_kNz = np.fft.fft2(p_top, axes=(0, 1))
    b = np.zeros(Nz - 1, dtype=np.complex128)
    c = np.zeros(Nz - 2, dtype=np.complex128)
    A = np.zeros((Nz - 1, Nz - 1), dtype=np.complex128)
    Kx, Ky = np.meshgrid(kx, ky, indexing='ij')
    # Repeat Kx and Ky for each z
    Kx = np.repeat(Kx[:, :, np.newaxis], Nz-2, axis=2)
    Ky = np.repeat(Ky[:, :, np.newaxis], Nz-2, axis=2)
    Gamma_right = 1 / dz ** 2 - rho_z / (2 * rho * dz) #+ 0 * 1j
    Gamma_left  = 1 / dz ** 2 + rho_z / (2 * rho * dz) #+ 0 * 1j
    # Gamma_rs = Kx ** 2 + Ky ** 2 - 2 / dz ** 2 - (Kx * rho_x + Ky * rho_y) / rho 
    Gamma_rs = -Kx ** 2 - Ky ** 2 - 2 / dz ** 2 - (Kx * rho_x * 1j + Ky * rho_y * 1j) / rho 
    # Gamma_rs = Kx * (Kx - rho_x / rho) + Ky * (Ky - rho_y/rho) - 2 / dz ** 2
    Gamma_right = 1 / dz ** 2 - rho_z / (rho * dz) #+ 0 * 1j
    Gamma_left  = np.ones_like(Kx) / dz ** 2 #+ rho_z / (2 * rho * dz) #+ 0 * 1j
    # Gamma_rs = Kx ** 2 + Ky ** 2 - 2 / dz ** 2 - (Kx * rho_x + Ky * rho_y) / rho 
    Gamma_rs = -Kx ** 2 - Ky ** 2 - 2 / dz ** 2 - (Kx * rho_x * 1j + Ky * rho_y * 1j - rho_z/dz) / rho 
    # asd = Kx ** 2 + Ky ** 2 
    # asd2 = (Kx * rho_x + Ky * rho_y) / rho
    # Solve the system for each gamma
    for r in range(Nx - 1):
        for s in range(Ny - 1):
            # Compute gammas
            # gamma_right = 1 / dz ** 2 - rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            # gamma_left  = 1 / dz ** 2 + rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            # gamma_rs = kx[r] ** 2 + ky[s] ** 2 - 2 / dz ** 2 - (kx[r] * rho_x[r, s] + ky[s] * rho_y[r, s]) / rho[r, s]
            # gamma_rs =  -(kx[r] ** 2 + ky[s] ** 2 + 2 / dz ** 2 + (kx[r] * rho_x[r, s] * 1j + ky[s] * rho_y[r, s] * 1j) / rho[r, s])
            gamma_right = Gamma_right[r, s]
            gamma_left = Gamma_left[r, s]
            gamma_rs = Gamma_rs[r, s]
            # Create RHS of system
            # F_k[r, s, 0] = dz * P_k0[r, s] + 0.5 * F_k[r, s, 1] / gamma_right[0] # dp/dz = 0
            F_k[r, s, 0] = P_k0[r, s] + 0.5 * F_k[r, s, 1] / (2 * dz * gamma_right[0]) # dp/dz = 0
            # Substract coefficient of top boundary condition
            F_k[r, s, -1] -=  P_kNz[r, s] * gamma_right[-1]
            # Create A in the system. Only keep diagonals of matrix
            a = gamma_left
            b[1:] = gamma_rs
            c[1:] = gamma_right[:-1]
            # Fix first coefficients
            # b[0] = 0.5 * gamma_left[0] / gamma_right[0] - 1.5
            # c[0] = 2 + 0.5 * gamma_rs[0] / gamma_right[0]
            b[0] =(-3 + gamma_left[0] / gamma_right[0]) / (2 * dz)
            c[0] = (4 + gamma_rs[0] / gamma_right[0]) / (2 * dz)
            # Create a tri-diagonal matrix A using a,b,c
            # A[0, 0] = b[0]
            # A[0, 1] = c[0]
            # for i in range(1, Nz - 2):
            #     A[i, i - 1] = a[i]
            #     A[i, i] = b[i]
            #     A[i, i + 1] = c[i]
            # A[-1, -2] = a[-1]
            # A[-1, -1] = b[-1]
            # print(np.linalg.cond(A))
            # up = 4 * rho_bk[r, s, 1] + rho_bk[r, s, 2] - rho_bk[r, s, 0]
            # do = 4 * rho_bk[r, s, 1] + rho_bk[r, s, 0] - rho_bk[r, s, 2]
            # b[0] = 0.5 * up / do - 1.5
            # tmp1 = gamma_left[0] / gamma_right[0]
            # tmp2 = up / do
            # err_tmp = np.abs(tmp1 - tmp2)
            # if err_tmp > 1e-10:
            #     print(tmp1, tmp2, err_tmp)
            # print(a.shape, b.shape, c.shape, gamma_left.shape, gamma_right.shape, gamma_rs.shape)
            # Solve system A P_k = F_k
            P_k[r, s, :] = thomas_algorithm(a, b, c, F_k[r, s, :])
            # P_k[r, s, :] = np.linalg.solve(A, F_k[r, s, :])
    # Compute IFFT in x-y slices to restore the 'physical' pressure
    p = np.real(np.fft.ifft2(P_k, axes=(0, 1)))
    # Add x boundary condition
    p = np.concatenate([p, np.expand_dims(p[0], axis=0)], axis=0)
    # Add y boundary condition
    p = np.concatenate([p, np.expand_dims(p[:, 0], axis=1)], axis=1)
    # Add top boundary condition
    p = np.concatenate([p, np.expand_dims(p_top, axis=2)], axis=2)
    return p#, (asd, asd2)

In [ ]:
def fftfd_density(f: np.ndarray, rho:np.ndarray, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> np.ndarray:
    Nx, Ny, Nz = x.shape[0], y.shape[0], z.shape[0]
    dx, dy, dz = x[1] - x[0], y[1] - y[0], z[1] - z[0]
    x_max = x[-1] # Max x value
    y_max = y[-1] # Max y value
    p_top = np.zeros((Nx, Ny)) # Top boundary condition
    F = f[:-1, :-1, :-1] # Removing boundaries and top boundary condition
    rr = np.fft.fftfreq(Nx - 1) * (Nx - 1) 
    ss = np.fft.fftfreq(Ny - 1) * (Ny - 1) 
    # Scale frequencies for any domain
    kx = 2 * np.pi * rr * 1j / x_max
    ky = 2 * np.pi * ss * 1j / y_max
    # Compute FFT in x direction (column-wise)
    F_k = np.fft.fft2(F, axes=(0, 1))
    P_k = np.zeros_like(F_k)
    # Compute gradient of density
    rho_x = np.gradient(rho, dx, axis=0, edge_order=2)
    rho_y = np.gradient(rho, dy, axis=1, edge_order=2)
    # rho_z = np.gradient(rho, dz, axis=2, edge_order=2)
    # rho_z = np.gradient(rho, 1, axis=2, edge_order=2)
    rho_z = np.roll(rho, -1, axis=2) - np.roll(rho, 1, axis=2)
    # Remove boundaries and top boundary
    rho_bk = rho
    rho = rho[:,:, 1:-1]
    rho_x = rho_x[:,:,1:-1]
    rho_y = rho_y[:,:,1:-1]
    rho_z = rho_z[:,:,1:-1]
    # Compute FFT in the last row (top boundary condition)
    P_kNz = np.fft.fft2(p_top, axes=(0, 1))
    b = np.zeros(Nz - 1, dtype=np.complex128)
    c = np.zeros(Nz - 2, dtype=np.complex128)
    rho_xd = (np.roll(rho_bk[:,:,1:-1], -1, axis=0) - np.roll(rho_bk[:,:,1:-1], 1, axis=0)) / (2 * dx)
    rho_yd = (np.roll(rho_bk[:,:,1:-1], -1, axis=1) - np.roll(rho_bk[:,:,1:-1], 1, axis=1)) / (2 * dy)
    # Solve the system for each gamma
    for r in range(Nx - 1):
        for s in range(Ny - 1):
            # Compute gammas
            # gamma_right = 1 / dz ** 2 - rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            # gamma_left  = 1 / dz ** 2 + rho_z[r, s] / (2 * rho[r, s] * dz) + 0 * 1j
            # gamma_rs = kx[r] ** 2 + ky[s] ** 2 - 2 / dz ** 2 - (kx[r] * rho_x[r, s] + ky[s] * rho_y[r, s]) / rho[r, s]
            gamma_right = 1 - (rho_bk[r, s, 2:] - rho_bk[r,s,:-2]) / (4 * rho[r, s]) + 0 * 1j
            gamma_left  = 1 + (rho_bk[r, s, 2:] - rho_bk[r,s,:-2]) / (4 * rho[r, s]) + 0 * 1j
            gamma_rs = (kx[r]*dz) ** 2 + (ky[s]*dz) ** 2 - 2  - dz ** 2 * (kx[r] * rho_xd[r,s] + ky[s] * rho_yd[r, s]) / rho[r, s]
            # # Create RHS of system
            # F_k[r, s, 0] = dz * 0 + 0.5 * F_k[r, s, 1] / gamma_right[0] # dp/dz = 0
            # # Substract coefficient of top boundary condition
            # F_k[r, s, -1] -=  P_kNz[r, s] * gamma_right[-1]
            # Create RHS of system
            F_k[r, s, 0] = dz * 0 + 0.5 * dz ** 2 * F_k[r, s, 1] / gamma_right[0] # dp/dz = 0
            # Substract coefficient of top boundary condition
            F_k[r, s, -1] = dz ** 2 * F_k[r, s, -1] -  P_kNz[r, s] * gamma_right[-1]
            F_k[r, s, 1:-1] = dz ** 2 * F_k[r, s, 1:-1]
            # Create A in the system. Only keep diagonals of matrix
            a = gamma_left
            b[1:] = gamma_rs
            c[1:] = gamma_right[:-1]
            # Fix first coefficients
            # b[0] = 0.5 * gamma_left[0] / gamma_right[0] - 1.5
            b[0] = 0.5 * (4 * rho_bk[r, s, 1] + rho_bk[r, s, 2] - rho_bk[r, s, 0]) / (4 * rho_bk[r, s, 1] + rho_bk[r, s, 0] - rho_bk[r, s, 2]) - 1.5
            c[0] = 2 + 0.5 * gamma_rs[0] / gamma_right[0]
            # Solve system A P_k = F_k
            P_k[r, s, :] = thomas_algorithm(a, b, c, F_k[r, s, :])
    # Compute IFFT in x-y slices to restore the 'physical' pressure
    p = np.real(np.fft.ifft2(P_k, axes=(0, 1)))
    # Add x boundary condition
    p = np.concatenate([p, np.expand_dims(p[0], axis=0)], axis=0)
    # Add y boundary condition
    p = np.concatenate([p, np.expand_dims(p[:, 0], axis=1)], axis=1)
    # Add top boundary condition
    p = np.concatenate([p, np.expand_dims(p_top, axis=2)], axis=2)
    return p

In [3]:
x0 = y0 = z0 = np.pi 
sx = sy = sz = .5 ** .5
A = .5

In [4]:
# # Functions
# G = lambda x, y, z: np.exp(-((x - x0) ** 2 / sx ** 2 + (y - y0) ** 2 / sy ** 2 + (z - z0) ** 2 / sz ** 2)) #+ 1
# Gx = lambda x, y, z: -2 * (x - x0) / sx ** 2 * G(x, y, z)
# Gxx = lambda x, y, z: (4 * (x - x0) ** 2 / sx ** 4 - 2 / sx ** 2) * G(x, y, z)
# Gy = lambda x, y, z: -2 * (y - y0) / sy ** 2 * G(x, y, z)
# Gyy = lambda x, y, z: (4 * (y - y0) ** 2 / sy ** 4 - 2 / sy ** 2) * G(x, y, z)
# Gz = lambda x, y, z: -2 * (z - z0) / sz ** 2 * G(x, y, z)
# Gzz = lambda x, y, z: (4 * (z - z0) ** 2 / sz ** 4 - 2 / sz ** 2) * G(x, y, z)
# rho = lambda x, y, z: A * G(x, y, z) 
# rhox = lambda x, y, z: A * Gx(x, y, z)
# rhoy = lambda x, y, z: A * Gy(x, y, z)
# rhoz = lambda x, y, z: A * Gz(x, y, z)
# g = lambda z: (2 * np.pi * z ** 2 - z ** 3) / np.pi ** 3 
# gz = lambda z: (4 * np.pi * z - 3 * z ** 2) / np.pi ** 3 
# gzz = lambda z: (4 * np.pi - 6 * z) / np.pi ** 3 
# p = lambda x, y, z: g(z) * G(x, y, z)
# px = lambda x, y, z: g(z) * Gx(x, y, z)
# py = lambda x, y, z: g(z) * Gy(x, y, z)
# pz = lambda x, y, z: g(z) * Gz(x, y, z) + gz(z) * G(x, y, z)
# pxx = lambda x, y, z: g(z) * Gxx(x, y, z)
# pyy = lambda x, y, z: g(z) * Gyy(x, y, z)
# pzz = lambda x, y, z: gzz(z) * G(x, y, z) + 2 * gz(z) * Gz(x, y, z) + g(z) * Gzz(x, y, z)
# f1 = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z) - (rhox(x, y, z) * px(x, y, z) + rhoy(x, y, z) * py(x, y, z) + rhoz(x, y, z) * pz(x, y, z)) / rho(x, y, z)
# f = lambda x, y, z: 2 * (z * (-3 + z  * (z / sx ** 2 + z / sy **2 + (4 * z - 3 * z0)/ sz ** 2)) + np.pi * (2 + 2 * z * (-z / sx ** 2 - z / sy ** 2 + (2 * z0 - 3*z)/ sz ** 2))) * G(x, y, z) / np.pi ** 3
# # f = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z)
# log_abs = lambda p, pa: np.log(np.abs(p - pa))

In [5]:
# # Functions
# G = lambda x, y, z: np.exp(-((x - x0) ** 2 / sx ** 2 + (y - y0) ** 2 / sy ** 2 + (z - z0) ** 2 / sz ** 2))
# tx = lambda x, y, z: -2 * (x - x0) / sx ** 2 
# txx = lambda x, y, z: (4 * (x - x0) ** 2 / sx ** 4 - 2 / sx ** 2)
# ty = lambda x, y, z: -2 * (y - y0) / sy ** 2 
# tyy = lambda x, y, z: (4 * (y - y0) ** 2 / sy ** 4 - 2 / sy ** 2) 
# tz = lambda x, y, z: -2 * (z - z0) / sz ** 2 
# tzz = lambda x, y, z: (4 * (z - z0) ** 2 / sz ** 4 - 2 / sz ** 2) 
# rho = lambda x, y, z: A + x * 0
# # rhox = lambda x, y, z: A * Gx(x, y, z)
# # rhoy = lambda x, y, z: A * Gy(x, y, z)
# # rhoz = lambda x, y, z: A * Gz(x, y, z)
# g = lambda z: (2 * np.pi * z ** 2 - z ** 3) / np.pi ** 3 
# gz = lambda z: (4 * np.pi * z - 3 * z ** 2) / np.pi ** 3 
# gzz = lambda z: (4 * np.pi - 6 * z) / np.pi ** 3 
# p = lambda x, y, z: g(z)
# px = lambda x, y, z: g(z) * tx(x, y, z)
# py = lambda x, y, z: g(z) * ty(x, y, z)
# pz = lambda x, y, z: g(z) * tz(x, y, z) + gz(z)
# pxx = lambda x, y, z: g(z) * txx(x, y, z)
# pyy = lambda x, y, z: g(z) * tyy(x, y, z)
# pzz = lambda x, y, z: gzz(z) + 2 * gz(z) * tz(x, y, z) + g(z) * tzz(x, y, z)
# f = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z) - (px(x, y, z) + py(x, y, z) + pz(x, y, z)) 
# #f = lambda x, y, z: 2 * (z * (-3 + z  * (z / sx ** 2 + z / sy **2 + (4 * z - 3 * z0)/ sz ** 2)) + np.pi * (2 + 2 * z * (-z / sx ** 2 - z / sy ** 2 + (2 * z0 - 3*z)/ sz ** 2))) * G(x, y, z) / np.pi ** 3
# # f = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z)
# log_abs = lambda p, pa: np.log(np.abs(p - pa)+1e-16)

In [6]:
# Functions
G = lambda x, y, z: np.exp(-((x - x0) ** 2 / sx ** 2 + (y - y0) ** 2 / sy ** 2 + (z - z0) ** 2 / sz ** 2)) 
Gx = lambda x, y, z: -2 * (x - x0) / sx ** 2 * G(x, y, z)
Gxx = lambda x, y, z: (4 * (x - x0) ** 2 / sx ** 4 - 2 / sx ** 2) * G(x, y, z)
Gy = lambda x, y, z: -2 * (y - y0) / sy ** 2 * G(x, y, z)
Gyy = lambda x, y, z: (4 * (y - y0) ** 2 / sy ** 4 - 2 / sy ** 2) * G(x, y, z)
Gz = lambda x, y, z: -2 * (z - z0) / sz ** 2 * G(x, y, z)
Gzz = lambda x, y, z: (4 * (z - z0) ** 2 / sz ** 4 - 2 / sz ** 2) * G(x, y, z)
rho = lambda x, y, z: A * G(x, y, z) + 1
rhox = lambda x, y, z: A * Gx(x, y, z)
rhoy = lambda x, y, z: A * Gy(x, y, z)
rhoz = lambda x, y, z: A * Gz(x, y, z)
# rho = lambda x, y, z: 1 + x * 0
# rhox = lambda x, y, z: x * 0
# rhoy = lambda x, y, z: x * 0
# rhoz = lambda x, y, z: x * 0
g = lambda z: (2 * np.pi * z ** 2 - z ** 3) / np.pi ** 3 
gz = lambda z: (4 * np.pi * z - 3 * z ** 2) / np.pi ** 3 
gzz = lambda z: (4 * np.pi - 6 * z) / np.pi ** 3 
p = lambda x, y, z: g(z) * G(x, y, z)
px = lambda x, y, z: g(z) * Gx(x, y, z)
py = lambda x, y, z: g(z) * Gy(x, y, z)
pz = lambda x, y, z: g(z) * Gz(x, y, z) + gz(z) * G(x, y, z)
pxx = lambda x, y, z: g(z) * Gxx(x, y, z)
pyy = lambda x, y, z: g(z) * Gyy(x, y, z)
pzz = lambda x, y, z: gzz(z) * G(x, y, z) + 2 * gz(z) * Gz(x, y, z) + g(z) * Gzz(x, y, z)
f = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z) - (rhox(x, y, z) * px(x, y, z) + rhoy(x, y, z) * py(x, y, z) + rhoz(x, y, z) * pz(x, y, z)) / rho(x, y, z)
#f = lambda x, y, z: 2 * (z * (-3 + z  * (z / sx ** 2 + z / sy **2 + (4 * z - 3 * z0)/ sz ** 2)) + np.pi * (2 + 2 * z * (-z / sx ** 2 - z / sy ** 2 + (2 * z0 - 3*z)/ sz ** 2))) * G(x, y, z) / np.pi ** 3
# f = lambda x, y, z: pxx(x, y, z) + pyy(x, y, z) + pzz(x, y, z)
log_abs = lambda p, pa: np.log(np.abs(p - pa))

In [7]:
def experiment(Nx, Ny, Nz):
    x = np.linspace(0, 2 * np.pi, Nx)
    y = np.linspace(0, 2 * np.pi, Ny)
    z = np.linspace(0, 2 * np.pi, Nz)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    P = p(X, Y, Z)
    F = f(X, Y, Z)
    RHO = rho(X, Y, Z)
    return x, y, z, P, F, RHO

In [8]:
def solve_iterative(x, y, z, p0, rho, f, tol=1e-14, max_iter=20, log=False):
    dx, dy, dz = x[1] - x[0], y[1] - y[0], z[1] - z[0]
    for m in range(1, max_iter+1):
        px = np.gradient(p0, dx, axis=0, edge_order=2)
        py = np.gradient(p0, dy, axis=1, edge_order=2)
        pz = np.gradient(p0, dz, axis=2, edge_order=2)
        rhox = np.gradient(rho, dx, axis=0, edge_order=2)
        rhoy = np.gradient(rho, dy, axis=1, edge_order=2)
        rhoz = np.gradient(rho, dz, axis=2, edge_order=2)
        f_new = f + (rhox * px + rhoy * py + rhoz * pz) / rho 
        p = fftfd(f_new, x, y, z)
        error = np.linalg.norm(p.flatten() - p0.flatten(), np.inf)
        p0 = p.copy()
        if log:
            print("m = %d, error = %.e" % (m, error))
        if error < tol:        
            break
    return p

In [9]:
N = 128
x, y, z, P, F, RHO = experiment(N, N, N)

### Direct Method

In [10]:
P_old = fftfd(F, x, y, z)
# error = np.linalg.norm(P_a.flatten() - P.flatten(), np.inf)
# error_base = np.linalg.norm(P_base.flatten() - P.flatten(), np.inf)
# print(error, error_base)
# plot(x, y, z, P*0 + P_a*1)

In [11]:
P_a = fftfd_density(F, RHO, x, y, z)

In [ ]:
np.linalg.norm(P_a.flatten() - P.flatten(), np.inf), np.linalg.norm(P_old.flatten() - P.flatten(), np.inf)

In [ ]:
(1j/10)**2

In [ ]:
plot(x, y, z, P_a)

### Iterative method

In [ ]:
P_a_iter = solve_iterative(x, y, z, np.zeros_like(P), RHO, F, tol=1e-16, max_iter=13, log=True)

In [ ]:
plot(x, y, z, P, title="Analytical solution")
plot(x, y, z, P_a_iter, title="Iterative solution")
print("Iterative error = %.e" % np.linalg.norm(P.flatten() - P_a_iter.flatten(), np.inf))
plot(x, y, z, P_a, title="Direct Solver")
print("Direct error = %.e" % np.linalg.norm(P.flatten() - P_a.flatten(), np.inf))

In [ ]:
np.argwhere(np.abs(P- P_a) == np.abs(P- P_a).max())

In [ ]:
error = np.linalg.norm(P_a.flatten() - P.flatten(), np.inf)
print(error)
plot(x, y, z, np.log(np.abs(P - P_old)/np.abs(P+1e-16)+1e-16), title="Abs error")

In [ ]:
Ns = np.array([16, 32, 64, 128, 256])
errors = np.zeros(len(Ns))
times = np.zeros(len(Ns))
dzs = np.zeros(len(Ns))
N_exp = 3
for i in range(len(Ns)):
    N = Ns[i]
    print("Computing for N =", N)
    x, y, z, P, F, RHO = experiment(N, N, N)
    start = time.time()
    for e in range(N_exp):
        # P_a = fftfd(F, x, y, z)
        P_a = solve_iterative(x, y, z, np.zeros_like(P), RHO, F, log=True)
    end = time.time()
    dzs[i] = z[1] - z[0]    
    errors[i] = np.linalg.norm(P.flatten() - P_a.flatten()) / np.linalg.norm(P.flatten())
    times[i] = (end - start) / N_exp

In [ ]:
plt.plot(Ns, times, 'x--')
plt.plot(Ns, Ns ** 3 * np.log(Ns) / 6e4, 'b-o', label=r'$O(n^3 \log(n))$')
plt.plot(Ns, Ns ** 2 * np.log(Ns) / 2e4, 'c-o', label=r'$O(n^2 \log(n))$')
plt.grid(True)
plt.yscale('log')
plt.xlabel(r'$n$')
plt.ylabel('Average Times (s)')
plt.xscale('log', base=2)
plt.legend()
plt.plot()

In [ ]:
plt.plot(dzs, errors, 'x--r')
plt.plot(dzs, 10 * dzs ** 2, 'b-o', label=r'$O(\Delta z^2)$')
plt.grid(True)
plt.yscale('log')
plt.xlabel(r'$\Delta z^2$')
plt.ylabel('Error')
plt.xscale('log', base=2)
plt.legend()
plt.plot()

For the presentation...

In [ ]:
print("$N$ & Time (s)  \\\\")
for i in range(len(Ns)):
    print("%d & $%.2f$ \\\\" % (Ns[i], times[i]))

print("N Time")
for i in range(len(Ns)):
    print("%d  %.6f" % (Ns[i], times[i]))

In [ ]:
print("$\Delta z$ & Error  \\\\")
for i in range(len(Ns)):
    print("$%.2f$ & $%.8f$ \\\\" % (dzs[i], errors[i]))
print("dz Error")
for i in range(len(Ns)):
    print("%.8f  %.8f" % (dzs[i], errors[i]))

---